In [ ]:
# Dataset Download & Feature Extraction

import os
import requests
import zipfile

import cv2
import pandas as pd
from pycocotools.coco import COCO
from tqdm import tqdm

In [ ]:
from tqdm import tqdm
import requests
import os
import zipfile

# Dataset Configuration

IMG_URL = "https://datasets-u2m.s3.eu-west-3.amazonaws.com/tomatOD_images.zip"
ANN_URL = "https://datasets-u2m.s3.eu-west-3.amazonaws.com/tomatOD_annotations.zip"

DATA_DIR = "tomato_data"
os.makedirs(DATA_DIR, exist_ok=True)

IMG_ZIP = os.path.join(DATA_DIR, "images.zip")
ANN_ZIP = os.path.join(DATA_DIR, "annotations.zip")

# Download Dataset

def download_with_progress(url, output_path):
    response = requests.get(url, stream=True)
    total_size = int(response.headers.get("content-length", 0))
    block_size = 1024

    with open(output_path, "wb") as f, tqdm(
        total=total_size,
        unit="B",
        unit_scale=True,
        desc=os.path.basename(output_path),
    ) as pbar:
        for data in response.iter_content(block_size):
            f.write(data)
            pbar.update(len(data))

if not os.path.exists(IMG_ZIP):
    print("Downloading images...")
    download_with_progress(IMG_URL, IMG_ZIP)
else:
    print("Images already downloaded.")

if not os.path.exists(ANN_ZIP):
    print("Downloading annotations...")
    download_with_progress(ANN_URL, ANN_ZIP)
else:
    print("Annotations already downloaded.")

# Extract

print("Extracting datasets...")
with zipfile.ZipFile(IMG_ZIP, "r") as z:
    z.extractall(DATA_DIR)

with zipfile.ZipFile(ANN_ZIP, "r") as z:
    z.extractall(DATA_DIR)

print("Download & extraction completed.")


In [11]:
# Path Configuration

ANNOTATIONS_PATH = "tomato_data/annotations/tomatOD_annotations/tomatOD_train.json"
IMAGES_PATH = "tomato_data/train"

OUTPUT_CSV = "data/tomato_RGB_HSV_ratio_dataset.csv"
os.makedirs("data", exist_ok=True)

# Load COCO

coco = COCO(ANNOTATIONS_PATH)
print(f"Images      : {len(coco.imgs)}")
print(f"Annotations : {len(coco.anns)}")

loading annotations into memory...
Done (t=0.02s)
creating index...
index created!
Images      : 222
Annotations : 1953


In [ ]:
# Feature Extraction (RGB, HSV, Ratio)

rows = []

for ann_id in tqdm(coco.getAnnIds(), desc="Processing annotations"):
    ann = coco.loadAnns(ann_id)[0]
    img_info = coco.loadImgs(ann["image_id"])[0]

    img_path = os.path.join(IMAGES_PATH, img_info["file_name"])
    img = cv2.imread(img_path)

    if img is None:
        continue

    x, y, w, h = map(int, ann["bbox"])
    crop = img[y:y + h, x:x + w]

    if crop.size == 0:
        continue

    # RGB
    B, G, R = crop.mean(axis=(0, 1))

    # HSV
    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    H, S, V = hsv.mean(axis=(0, 1))

    # Ratio
    rg_ratio = R / (G + 1e-5)
    rb_ratio = R / (B + 1e-5)

    label = coco.loadCats(ann["category_id"])[0]["name"]

    rows.append([
        img_info["file_name"],
        R, G, B,
        H, S, V,
        rg_ratio, rb_ratio,
        label
    ])


Processing annotations: 100%|██████████| 1953/1953 [00:36<00:00, 53.81it/s]


In [ ]:
# Create DataFrame & Save

columns = [
    "Image",
    "R", "G", "B",
    "H", "S", "V",
    "R/G", "R/B",
    "Label"
]

df = pd.DataFrame(rows, columns=columns)
df.to_csv(OUTPUT_CSV, index=False)

print(f"Dataset saved to: {OUTPUT_CSV}")
print("Total samples :", len(df))
print("\nClass distribution:")
print(df["Label"].value_counts())


Dataset saved to: data/tomato_RGB_HSV_ratio_dataset.csv
Total samples : 1953

Class distribution:
Label
unripe        1301
fully-ripe     332
semi-ripe      320
Name: count, dtype: int64
